# Week 13, Day 4: Inequality constraints, sparsity and the L1 penalty

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w013/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Excitatory weights by projected gradient

A cell receives 20 excitatory inputs. Fit their non-negative weights from 200 noisy trials by projected gradient descent (5,000 steps, step size 1/L with L the largest eigenvalue of BᵀB). Store the result in `v`, the unconstrained least-squares fit in `w_ls`, and check the KKT conditions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1996)
B = rng.normal(0, 1, (200, 20))                 # presynaptic activity on each trial
w_true = np.maximum(rng.normal(0, 1, 20), 0)    # excitatory weights, many exactly zero
t = B @ w_true + 0.5 * rng.normal(size=200)     # postsynaptic response

L = np.linalg.norm(B, 2) ** 2
v = np.zeros(20)
for _ in range(5000):
    v = ...
w_ls = np.linalg.lstsq(B, t, rcond=None)[0]
g = B.T @ (B @ v - t)
print("zero weights:", np.sum(v == 0), " least-squares negatives:", np.sum(w_ls < 0))
print("largest |gradient| on positive weights:", np.abs(g[v > 0]).max(), " smallest gradient on zero weights:", g[v == 0].min().round(3))

In [ ]:
import numpy as np
def test_nonneg():
    "all fitted weights are non-negative"
    assert np.all(v >= 0)

def test_kkt():
    "KKT holds: zero gradient where v > 0, non-negative gradient where v = 0"
    g = B.T @ (B @ v - t)
    assert np.abs(g[v > 0]).max() < 1e-6 and g[v == 0].min() > -1e-6

def test_ls_negative():
    "unconstrained least squares gives some negative weights"
    assert np.sum(w_ls < 0) >= 1

test_nonneg()
test_kkt()
test_ls_negative()
print("All checks passed")

### Exercise: Soft thresholding

Write `soft(z, tau)` for arrays.

In [ ]:
def soft(z, tau):
    return ...

print(soft(np.array([-3.0, -0.5, 0.0, 0.8, 2.5]), 1.0))

In [ ]:
import numpy as np
def test_values():
    "soft(z, 1) shrinks by 1 and zeros the middle"
    assert np.allclose(soft(np.array([-3.0, -0.5, 0.0, 0.8, 2.5]), 1.0), [-2, 0, 0, 0, 1.5])

def test_minimises():
    "soft(z, tau) minimises 0.5 (w - z)^2 + tau |w| on a fine grid"
    ws = np.linspace(-5, 5, 100001)
    for z in [-2.2, 0.3, 1.7]:
        obj = 0.5 * (ws - z) ** 2 + 0.6 * np.abs(ws)
        assert abs(ws[np.argmin(obj)] - soft(np.array([z]), 0.6)[0]) < 1e-3

test_values()
test_minimises()
print("All checks passed")

### Exercise: Find the five inputs from sixty measurements

Write `ista(A, y, lam, steps)`. Recover a 5-sparse weight vector of length 100 from 60 noisy measurements with lam = 0.02 and 3,000 steps. Compare with the minimum-norm solution `w_mn = np.linalg.pinv(A) @ y`.

In [ ]:
rng = np.random.default_rng(1996)
n, p, k = 60, 100, 5
A = rng.normal(0, 1, (n, p)) / np.sqrt(n)
w_true = np.zeros(p)
support = rng.choice(p, k, replace=False)
w_true[support] = rng.choice([-1, 1], k) * rng.uniform(1, 2, k)
y = A @ w_true + 0.01 * rng.normal(size=n)

def ista(A, y, lam, steps):
    L = np.linalg.norm(A, 2) ** 2
    w = np.zeros(A.shape[1])
    for _ in range(steps):
        w = ...
    return w

w_ista = ista(A, y, 0.02, 3000)
w_mn = np.linalg.pinv(A) @ y
print("true inputs:", sorted(support), " found:", np.flatnonzero(np.abs(w_ista) > 0.1))
print("largest error, ISTA:", np.abs(w_ista - w_true).max().round(3), " minimum norm:", np.abs(w_mn - w_true).max().round(3))
plt.stem(w_true, linefmt="k-", markerfmt="ko", label="true"); plt.plot(w_ista, "o", label="ISTA"); plt.plot(w_mn, ".", label="minimum norm"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_support():
    "ISTA's weights larger than 0.1 are exactly the five true inputs"
    assert set(np.flatnonzero(np.abs(w_ista) > 0.1)) == set(support)

def test_accurate():
    "ISTA's weights are within 0.1 of the truth"
    assert np.abs(w_ista - w_true).max() < 0.1

def test_minnorm_dense():
    "the minimum-norm least-squares solution is dense and wrong"
    assert np.sum(np.abs(w_mn) > 0.1) > 20

test_support()
test_accurate()
test_minnorm_dense()
print("All checks passed")